In [12]:
import os, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

warnings.filterwarnings("ignore")

BASE = os.getcwd()   # Changed for notebook (no __file__)
RAW  = os.path.join(BASE, "aviation_combined_dataset.csv")
OUT  = os.path.join(BASE, "preprocessed_dataset.csv")

SEP  = "=" * 65

In [13]:
print(f"\n{SEP}")
print("STEP 1 — RAW AUDIT")
print(SEP)

df_raw = pd.read_csv(RAW)
print(f"Raw shape : {df_raw.shape}")

null_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)

print("\nColumn inventory (sorted by missing %):\n")
print(f"  {'Column':<45} {'Null%':>6}  {'Dtype':<12}  {'Unique':>7}")
print(f"  {'-'*45} {'-'*6}  {'-'*12}  {'-'*7}")

for col, pct in null_pct.items():
    flag = " ◀ DROP" if pct >= 90 else (" ◀ SPARSE" if pct >= 60 else "")
    print(f"  {col:<45} {pct:6.1f}%  {str(df_raw[col].dtype):<12}  {df_raw[col].nunique():>7}{flag}")
print("Columns loaded:", len(df_raw.columns))
print(df_raw.columns.tolist())


STEP 1 — RAW AUDIT
Raw shape : (2649, 62)

Column inventory (sorted by missing %):

  Column                                         Null%  Dtype          Unique
  --------------------------------------------- ------  ------------  -------
  live                                           100.0%  float64             0 ◀ DROP
  aircraft                                       100.0%  float64             0 ◀ DROP
  arrival.actual                                 100.0%  float64             0 ◀ DROP
  arrival.estimated_runway                       100.0%  float64             0 ◀ DROP
  arrival.actual_runway                          100.0%  float64             0 ◀ DROP
  flight.codeshared                              100.0%  float64             0 ◀ DROP
  live.updated                                    96.8%  str                52 ◀ DROP
  live.latitude                                   96.8%  float64            86 ◀ DROP
  live.longitude                                  96.8%  float64       

In [14]:
print(f"\n{SEP}")
print("STEP 2 — REMOVE SYNTHETIC / DISTANCE-REFERENCE ROWS")
print(SEP)

df = df_raw[df_raw["flight_status"] != "distance_reference"].copy()

print(f"Before : {len(df_raw):,} rows")
print(f"Removed: {len(df_raw) - len(df):,} distance_reference rows")
print(f"After  : {len(df):,} rows")


STEP 2 — REMOVE SYNTHETIC / DISTANCE-REFERENCE ROWS
Before : 2,649 rows
Removed: 149 distance_reference rows
After  : 2,500 rows


In [15]:
print(f"\n{SEP}")
print("STEP 3 — DROP COLUMNS (Detailed Report)")
print(SEP)

def drop_reason(col, series, total):
    pct = series.isnull().sum() / total * 100
    
    if pct == 100: return "100% null"
    if pct >= 90: return ">90% null"
    if pct >= 60: return ">60% null"

    leakage_keywords = ["actual", "estimated_runway", "estimated"]
    if any(kw in col for kw in leakage_keywords):
        return "post-flight leakage"

    if col.endswith(".icao") or col.endswith(".icao24"):
        return "ICAO duplicate"

    if "codeshared." in col:
        return "codeshare duplicate"

    id_exact = {
        "flight.codeshared","aircraft","live","flight_status",
        "departure.airport","arrival.airport",
        "departure.timezone","arrival.timezone",
        "flight.number","flight.iata","flight.icao",
        "airline.icao","airline.name",
        "terminal_from","terminal_to"
    }

    if col in id_exact:
        return "identifier / no signal"

    return None


# 🔹 BEFORE
total_before = df.shape[1]
print(f"\nTotal columns BEFORE dropping: {total_before}\n")

dropped = {}

for col in list(df.columns):
    reason = drop_reason(col, df[col], len(df))
    if reason:
        dropped[col] = reason

# 🔹 DROP
df.drop(columns=list(dropped.keys()), inplace=True)

# 🔹 AFTER
total_after = df.shape[1]
print(f"\nTotal columns AFTER dropping : {total_after}")
print(f"Total columns REMOVED        : {len(dropped)}\n")


# 🔹 PRINT DROPPED COLUMNS
print("Dropped Columns (with reasons):\n")
print(f"{'Column Name':<50} {'Reason'}")
print("-" * 70)

for col, reason in dropped.items():
    print(f"{col:<50} {reason}")


# 🔹 PRINT REMAINING COLUMNS
print("\nRemaining Columns:\n")
for col in df.columns:
    print(f"✔ {col}")


STEP 3 — DROP COLUMNS (Detailed Report)

Total columns BEFORE dropping: 62


Total columns AFTER dropping : 15
Total columns REMOVED        : 47

Dropped Columns (with reasons):

Column Name                                        Reason
----------------------------------------------------------------------
flight_status                                      identifier / no signal
aircraft                                           100% null
live                                               100% null
departure.airport                                  identifier / no signal
departure.timezone                                 identifier / no signal
departure.icao                                     ICAO duplicate
departure.delay                                    >60% null
departure.estimated                                post-flight leakage
departure.actual                                   >60% null
departure.estimated_runway                         >60% null
departure.actual_runway    

In [16]:
df.head()

,flight_date,departure.iata,departure.terminal,departure.gate,departure.scheduled,arrival.iata,arrival.terminal,arrival.scheduled,airline.iata,terminal_distance_m,terminal_walk_time_min,terminal_transport_method,connection_time_min,arrival_delay_min,connection_risk
0,2026-04-26,ORD,2,F25,2026-04-26T22:45:00+00:00,SGF,E,2026-04-27T00:36:00+00:00,LH,2000,10,Shuttle Bus,69.0,16.0,Risky
1,2026-04-27,MUC,2,G20,2026-04-27T10:20:00+00:00,ZAG,NaN,2026-04-27T11:25:00+00:00,UA,400,5,Walkway Bridge,96.0,26.0,Tight
2,2026-04-27,MUC,2,G,2026-04-27T10:15:00+00:00,HAM,2,2026-04-27T11:30:00+00:00,AZ,400,5,Walkway Bridge,75.0,6.0,Tight
3,2026-04-26,LAS,1,C23,2026-04-26T22:50:00+00:00,SMF,B,2026-04-27T00:25:00+00:00,WN,400,5,Walkway,65.0,11.0,Tight
4,2026-04-27,CDG,2F,F05,2026-04-27T09:15:00+00:00,VIE,1,2026-04-27T11:20:00+00:00,VN,1200,8,Shuttle Bus,112.0,22.0,Tight


In [17]:
print(f"\n{SEP}")
print("STEP 4 — TIME FEATURES")
print(SEP)

# Convert to datetime
df["dep_sched"] = pd.to_datetime(df["departure.scheduled"], utc=True, errors="coerce")
df["arr_sched"] = pd.to_datetime(df["arrival.scheduled"], utc=True, errors="coerce")

# Create features
df["scheduled_flight_min"] = (df["arr_sched"] - df["dep_sched"]).dt.total_seconds() / 60
df["departure_hour"] = df["dep_sched"].dt.hour
df["day_of_week"] = df["dep_sched"].dt.dayofweek

# Remove invalid rows
bad = df["scheduled_flight_min"] <= 0
df = df[~bad].copy()

# 🔥 PRINT OUTPUT (THIS WAS MISSING)
print(f"\nRows removed (invalid duration): {bad.sum()}")
print(f"Shape after STEP 4: {df.shape}")

print("\nFeature Summary:")
print(f"departure_hour range : {df['departure_hour'].min()} → {df['departure_hour'].max()}")
print(f"day_of_week range    : {df['day_of_week'].min()} → {df['day_of_week'].max()}")
print(f"avg flight duration  : {df['scheduled_flight_min'].mean():.2f} mins")

print("\nSample Data:")
print(df[[
    "departure.scheduled",
    "arrival.scheduled",
    "scheduled_flight_min",
    "departure_hour",
    "day_of_week"
]].head())


STEP 4 — TIME FEATURES

Rows removed (invalid duration): 1
Shape after STEP 4: (2499, 20)

Feature Summary:
departure_hour range : 0 → 23
day_of_week range    : 0 → 6
avg flight duration  : 165.32 mins

Sample Data:
         departure.scheduled          arrival.scheduled  scheduled_flight_min  \
0  2026-04-26T22:45:00+00:00  2026-04-27T00:36:00+00:00                 111.0   
1  2026-04-27T10:20:00+00:00  2026-04-27T11:25:00+00:00                  65.0   
2  2026-04-27T10:15:00+00:00  2026-04-27T11:30:00+00:00                  75.0   
3  2026-04-26T22:50:00+00:00  2026-04-27T00:25:00+00:00                  95.0   
4  2026-04-27T09:15:00+00:00  2026-04-27T11:20:00+00:00                 125.0   

   departure_hour  day_of_week  
0              22            6  
1              10            0  
2              10            0  
3              22            6  
4               9            0  


In [18]:
print(f"\n{SEP}")
print("STEP 5 — TARGET")
print(SEP)

RISK_MAP = {"Risky":0, "Tight":1, "Safe":2}

df["risk_label"] = df["connection_risk"].map(RISK_MAP)
df.drop(columns=["connection_risk"], inplace=True)

print(df["risk_label"].value_counts())


STEP 5 — TARGET
risk_label
2    999
0    750
1    750
Name: count, dtype: int64


In [19]:
print(f"\n{SEP}")
print("STEP 6 — FEATURE ENGINEERING (ADVANCED DYNAMIC)")
print(SEP)

import pandas as pd
import numpy as np
import random

# ─────────────────────────────────────────────
# 6a. Transport friction
# ─────────────────────────────────────────────
FRICTION = {
    "walkway": 1,
    "walkway bridge": 1,
    "moving walkway": 1,
    "airside connector": 2,
    "skytrain": 2,
    "monorail": 2,
    "automated people mover": 2,
    "train": 2,
    "shuttle": 3,
    "shuttle bus": 3,
    "bus": 3,
}

def friction_score(val):
    if pd.isna(val): return 3
    v = str(val).lower()
    for k, s in FRICTION.items():
        if k in v:
            return s
    return 3

df["transport_friction"] = df["terminal_transport_method"].apply(friction_score)


# ─────────────────────────────────────────────
# 6b. Airport flags
# ─────────────────────────────────────────────
df["is_cross_airport"] = (df["departure.iata"] != df["arrival.iata"]).astype(int)

INTL_HUBS = {"DXB","LHR","SIN","AMS","FRA","CDG","JFK","NRT","IST","HKG","ORD","LAX"}
df["is_intl_hub"] = df["departure.iata"].isin(INTL_HUBS).astype(int)

# Major busy airports (extra crowd boost)
MAJOR_AIRPORTS = {"LAX","JFK","LHR","CDG","DXB","SIN","AMS","FRA","ORD"}
df["is_major_airport"] = df["departure.iata"].isin(MAJOR_AIRPORTS).astype(int)


# ─────────────────────────────────────────────
# 6c. Peak hour
# ─────────────────────────────────────────────
df["is_peak_hour"] = df["departure_hour"].apply(
    lambda h: 1 if (6 <= h <= 10 or 17 <= h <= 21) else 0
)


# ─────────────────────────────────────────────
# 6d. Terminal change
# ─────────────────────────────────────────────
df["terminal_change"] = (
    df["departure.terminal"].fillna("?") != df["arrival.terminal"].fillna("??")
).astype(int)


# ─────────────────────────────────────────────
# 6e. Delay info flag
# ─────────────────────────────────────────────
df["has_delay_info"] = df["arrival_delay_min"].notna().astype(int)


# ─────────────────────────────────────────────
# 6f. BASE CROWD FROM TIME
# ─────────────────────────────────────────────
def base_crowd(h):
    if 6 <= h <= 10 or 17 <= h <= 21:
        return 2   # high
    elif 10 < h < 17:
        return 1   # medium
    else:
        return 0   # low


# ─────────────────────────────────────────────
# 6g. ADVANCED CROWD CALCULATION
# ─────────────────────────────────────────────
def compute_crowd(row, crowd_type="security"):
    crowd = base_crowd(row["departure_hour"])

    # 🟠 Airport size effect
    if row["is_major_airport"] == 1:
        crowd += 0.5

    # 🟠 Day of week effect
    # 0=Mon ... 6=Sun
    if row["day_of_week"] in [4, 5, 6]:  # Fri, Sat, Sun
        crowd += 0.5

    # 🟠 Random noise (real-world unpredictability)
    crowd += random.uniform(-0.3, 0.3)

    # 🟠 Separate logic for immigration (usually slower)
    if crowd_type == "immigration":
        crowd += 0.3

    # Normalize to levels
    if crowd < 0.75:
        return "low"
    elif crowd < 1.75:
        return "medium"
    else:
        return "high"


df["security_crowd"] = df.apply(lambda r: compute_crowd(r, "security"), axis=1)
df["immigration_crowd"] = df.apply(lambda r: compute_crowd(r, "immigration"), axis=1)


# ─────────────────────────────────────────────
# 6h. Crowd multiplier
# ─────────────────────────────────────────────
def crowd_multiplier(level):
    if level == "low":
        return 0.8
    elif level == "medium":
        return 1.0
    else:
        return 1.5


# ─────────────────────────────────────────────
# 6i. Time components
# ─────────────────────────────────────────────
df["walking_time_min"] = df["terminal_walk_time_min"]

BASE_SECURITY = 20
BASE_IMMIGRATION = 30

df["security_time_min"] = df["security_crowd"].apply(
    lambda c: BASE_SECURITY * crowd_multiplier(c)
)

df["immigration_time_min"] = df.apply(
    lambda row: BASE_IMMIGRATION * crowd_multiplier(row["immigration_crowd"])
    if row["is_intl_hub"] == 1 else 0,
    axis=1
)

# congestion (also dynamic)
df["congestion_time_min"] = df["departure_hour"].apply(
    lambda h: 15 if (6 <= h <= 10 or 17 <= h <= 21)
    else 8 if 10 < h < 17
    else 3
)

# baggage
df["baggage_time_min"] = 10


# ─────────────────────────────────────────────
# 6j. Total required time
# ─────────────────────────────────────────────
df["required_time_min"] = (
    df["walking_time_min"]
    + df["security_time_min"]
    + df["immigration_time_min"]
    + df["congestion_time_min"]
    + df["baggage_time_min"]
)


# ─────────────────────────────────────────────
# 6k. Time margin (core feature)
# ─────────────────────────────────────────────
df["time_margin_min"] = (
    df["connection_time_min"]
    - df["arrival_delay_min"]
    - df["required_time_min"]
)


# ─────────────────────────────────────────────
# OUTPUT SUMMARY
# ─────────────────────────────────────────────
print("\nEngineered Features Summary:\n")

features = [
    "transport_friction",
    "is_cross_airport",
    "is_intl_hub",
    "is_major_airport",
    "is_peak_hour",
    "terminal_change",
    "security_crowd",
    "immigration_crowd",
    "required_time_min",
    "time_margin_min"
]

for f in features:
    if pd.api.types.is_numeric_dtype(df[f]):
        print(f"{f:25s} → mean={df[f].mean():.2f}")
    else:
        print(f"{f:25s} → sample={df[f].unique()[:3]}")

print("\nFinal Shape:", df.shape)


STEP 6 — FEATURE ENGINEERING (ADVANCED DYNAMIC)

Engineered Features Summary:

transport_friction        → mean=1.93
is_cross_airport          → mean=0.97
is_intl_hub               → mean=0.24
is_major_airport          → mean=0.18
is_peak_hour              → mean=0.49
terminal_change           → mean=0.85
security_crowd            → sample=<StringArray>
['medium', 'high', 'low']
Length: 3, dtype: str
immigration_crowd         → sample=<StringArray>
['medium', 'high', 'low']
Length: 3, dtype: str
required_time_min         → mean=62.62
time_margin_min           → mean=15.37

Final Shape: (2499, 36)


In [20]:
print(f"\n{SEP}")
print("STEP 7 — MISSING VALUES")
print(SEP)

# Categorical → fill with "Unknown"
for col in df.select_dtypes(include="object"):
    nulls = df[col].isnull().sum()
    if nulls > 0:
        df[col] = df[col].fillna("Unknown")
        print(f"{col}: filled {nulls} nulls")

# Numeric → fill with median
for col in df.select_dtypes(include=np.number):
    nulls = df[col].isnull().sum()
    if nulls > 0:
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f"{col}: filled {nulls} nulls with median")


STEP 7 — MISSING VALUES
departure.terminal: filled 516 nulls
departure.gate: filled 653 nulls
arrival.terminal: filled 639 nulls
airline.iata: filled 20 nulls


In [21]:
df.to_csv("cleaned_dataset_before_encoding.csv", index=False)